# Detecção de Gestos em Tempo Real com MediaPipe

Este notebook utiliza o modelo `gesture_recognizer.task` do **Google MediaPipe** para reconhecer gestos manuais através da webcam em tempo real.

### Pré-requisitos
Certifique-se de que as dependências estão instaladas:
```bash
pip install mediapipe opencv-python numpy
```

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import time

## Configuração do Reconhecedor de Gestos

Configuramos o modelo para o modo de vídeo, permitindo o processamento contínuo de frames da webcam.

In [2]:
model_path = 'models/gesture_recognizer.task'

BaseOptions = mp.tasks.BaseOptions
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

recognizer = GestureRecognizer.create_from_options(options)

## Funções Utilitárias para Visualização

Função para desenhar os pontos das mãos (landmarks) e o nome do gesto detectado no frame usando OpenCV diretamente, evitando problemas com o módulo `mp.solutions`.

In [3]:
# Conexões das juntas da mão para desenho manual
HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),      # Polegar
    (0, 5), (5, 6), (6, 7), (7, 8),      # Indicador
    (5, 9), (9, 10), (10, 11), (11, 12),  # Médio
    (9, 13), (13, 14), (14, 15), (15, 16), # Anelar
    (0, 17), (17, 18), (18, 19), (19, 20), # Mindinho
    (5, 9), (9, 13), (13, 17)             # Base da palma
]

def draw_gesture_results(image, gesture_recognition_result):
    """Desenha landmarks e nomes dos gestos sem usar mp.solutions."""
    if not gesture_recognition_result or not gesture_recognition_result.hand_landmarks:
        return image

    for i, hand_landmarks in enumerate(gesture_recognition_result.hand_landmarks):
        h, w, _ = image.shape
        
        # Converte landmarks normalizados para coordenadas de pixels
        points = []
        for lm in hand_landmarks:
            points.append((int(lm.x * w), int(lm.y * h)))

        # Desenha as conexões (juntas)
        for start, end in HAND_CONNECTIONS:
            cv2.line(image, points[start], points[end], (0, 255, 0), 2)

        # Desenha os pontos (articulações)
        for p in points:
            cv2.circle(image, p, 5, (255, 0, 255), -1)

        # Verifica se há gestos reconhecidos para esta mão
        if gesture_recognition_result.gestures:
            gestures = gesture_recognition_result.gestures[i]
            if gestures:
                top_gesture = gestures[0]
                gesture_name = top_gesture.category_name
                score = round(top_gesture.score, 2)
                
                # Exibe o nome do gesto acima da posição média da mão
                avg_x = sum([p[0] for p in points]) // len(points)
                avg_y = min([p[1] for p in points]) - 30
                
                # Desenha contorno preto para maior contraste
                cv2.putText(image, f"{gesture_name} ({score})", (avg_x - 50, avg_y),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4, cv2.LINE_AA)
                # Desenha o texto branco por cima
                cv2.putText(image, f"{gesture_name} ({score})", (avg_x - 50, avg_y),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)

    return image

## Loop de Processamento da Webcam

Captura o vídeo, processa cada quadro com o modelo e exibe o resultado.

In [4]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Erro: Não foi possível abrir a webcam.")
else:
    print("Webcam aberta com sucesso! Pressione 'q' para sair.")
    
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("Falha ao capturar imagem.")
            break

        # Inverte o frame horizontalmente para uma experiência mais natural
        frame = cv2.flip(frame, 1)

        # Converte para RGB (exigido pelo MediaPipe)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # Timestamp em milissegundos
        timestamp_ms = int(time.time() * 1000)

        # Realiza o reconhecimento de gestos
        recognition_result = recognizer.recognize_for_video(mp_image, timestamp_ms)

        # Desenha resultados no frame original
        annotated_frame = draw_gesture_results(frame.copy(), recognition_result)

        # Mostra o resultado
        cv2.imshow('MediaPipe Gesture Recognition', annotated_frame)

        # Sai do loop se 'q' for pressionado
        if cv2.waitKey(5) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    recognizer.close()
    print("Webcam encerrada.")

Webcam aberta com sucesso! Pressione 'q' para sair.
Webcam encerrada.
